In [ ]:
from etils.etree import jax
%cd /content
!git clone https://github.com/Matiata/maxim.git
%cd /content/maxim
!pip install -r requirements.txt
!pip install -e .
# %rm -rf /content/maxim

/content
Cloning into 'maxim'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 283 (delta 128), reused 123 (delta 75), pack-reused 100 (from 1)
Receiving objects: 100% (283/283), 15.37 MiB | 16.89 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/maxim
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.6 MB/s eta 0:00:00
Obtaining file:///content/maxim
  Preparing metadata (setup.py) ... done
  Running setup.py develop for maxim


In [ ]:
from google.colab import drive # works only for colab
drive.mount('/content/gdrive/',)

Mounted at /content/gdrive/


In [ ]:
import importlib
import collections
import ml_collections
import io
import jax.numpy as jnp
from jax import random
import numpy as np
import tensorflow as tf
from flax.core import freeze, unfreeze
from flax.traverse_util import flatten_dict, unflatten_dict

SEED = 42
NUM_EXPERTS = 5
CKPT_PATH = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/ckpt_Enhancement_LOL.npz"
TASK = "enhance"
_MODEL_CONFIGS = {
    "variant": "",
    "dropout_rate": 0.0,
    "num_outputs": 3,
    "use_bias": True,
    "num_supervision_scales": 3,
}
_MODEL_VARIANT_DICT = {
    "denoise": "S-3",
    "deblur": "S-3",
    "derain": "S-2",
    "dehaze": "S-2",
    "enhance": "S-2",
}

# Trainer helpers

In [ ]:
class TrainState(train_state.TrainState):
    """Extended train state with batch statistics."""

    batch_stats: Any = None

def random_crop(image, target, crop_size):
    """Random crop for data augmentation."""
    h, w = image.shape[0], image.shape[1]

    if h > crop_size and w > crop_size:
        top = np.random.randint(0, h - crop_size)
        left = np.random.randint(0, w - crop_size)

        image = image[top : top + crop_size, left : left + crop_size]
        target = target[top : top + crop_size, left : left + crop_size]

    return image, target

def random_flip(image, target):
    """Random horizontal and vertical flip."""
    if np.random.rand() > 0.5:
        image = np.fliplr(image)
        target = np.fliplr(target)

    if np.random.rand() > 0.5:
        image = np.flipud(image)
        target = np.flipud(target)

    return image, target

def random_rotation(image, target):
    """Random 90-degree rotation."""
    k = np.random.randint(0, 4)
    image = np.rot90(image, k=k)
    target = np.rot90(target, k=k)
    return image, target

def read_lines_from_file(basepath, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    existing = []
    missing = []

    for line in lines:
        p = os.path.join(basepath, line)
        if os.path.exists(p):
            existing.append(p)
        else:
            missing.append(p)

    print(
        f"Checked {len(lines)} files in {basepath}: {len(existing)} existing, {len(missing)} missing."
    )

    return existing, missing

def load_image(filepath):
    """Load and preprocess image."""
    img = Image.open(filepath).convert("RGB")
    img = np.asarray(img, np.float32) / 255.0
    return img

def create_dataset(data_dir, batch_size, patch_size, is_training=True):
    """Create TensorFlow dataset for training/validation."""

    print(
        f"Creating {'training' if is_training else 'validation'} dataset from {data_dir}"
    )
    input_dir = (
        os.path.join(data_dir, "train")
        if is_training
        else os.path.join(data_dir, "test")
    )
    target_dir = os.path.join(data_dir, "GT")
    files_list = (
        os.path.join(data_dir, "train.txt")
        if is_training
        else os.path.join(data_dir, "test.txt")
    )

    input_files, _ = read_lines_from_file(input_dir, files_list)
    target_files, _ = read_lines_from_file(target_dir, files_list)

    def load_and_preprocess(input_path, target_path):
        """Load and preprocess a single pair of images."""
        input_img = load_image(input_path.numpy().decode())
        target_img = load_image(target_path.numpy().decode())

        orig_h, orig_w = input_img.shape[:2]

        # Padding images to have even shapes
        input_img = make_shape_even(input_img)
        target_img = make_shape_even(target_img)
        even_h, even_w = input_img.shape[:2]

        # Padding images to be multiples of 64
        input_img = mod_padding_symmetric(input_img, factor=64)
        target_img = mod_padding_symmetric(target_img, factor=64)
        pad_h, pad_w = input_img.shape[:2]

        if is_training:
            # Data augmentation
            input_img, target_img = random_crop(input_img, target_img, patch_size)
            input_img, target_img = random_flip(input_img, target_img)
            input_img, target_img = random_rotation(input_img, target_img)

        return (
            input_img.astype(np.float32),
            target_img.astype(np.float32),
            np.array([orig_h, orig_w, even_h, even_w, pad_h, pad_w], np.int32),
        )

    dataset = tf.data.Dataset.from_tensor_slices((input_files, target_files))

    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.map(
        lambda x, y: tf.py_function(
            func=load_and_preprocess,
            inp=[x, y],
            Tout=[tf.float32, tf.float32, tf.int32],
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.batch(batch_size, drop_remainder=is_training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, len(input_files)

def resize_target_to(pred, target):
    """Downsample target if necessary to match prediction shape."""
    if pred.shape == target.shape:
        return target
    # Compute integer stride factors
    scale_h = target.shape[1] // pred.shape[1]
    scale_w = target.shape[2] // pred.shape[2]
    # Subsample target by simple stride (fast, deterministic)
    return target[:, ::scale_h, ::scale_w, :]

def compute_psnr(pred, target):
    """Compute PSNR metric.

    Args:
      pred: Predicted image in [0, 1] range
      target: Target image in [0, 1] range

    Returns:
      PSNR in dB, computed as if images were in [0, 255] range
      to match the evaluation script.
    """
    # Convert to [0, 255] range for PSNR calculation
    pred_255 = pred * 255.0
    target_255 = target * 255.0

    mse = jnp.mean((pred_255 - target_255) ** 2)
    mse = jnp.maximum(mse, 1e-6)  # Avoid division by zero

    psnr = 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))
    return psnr

def router_entropy_loss(probs, eps=1e-8):
    return -jnp.mean(jnp.sum(probs * jnp.log(probs + eps), axis=-1))

def load_balance_loss(probs):
    usage = jnp.mean(probs, axis=0)
    num_experts = probs.shape[-1]
    return num_experts * jnp.sum(usage ** 2)

def compute_loss_maxim(preds, targets, num_scales=3):
    """Compute multi-scale L1 loss."""
    total_loss = 0.0

    if isinstance(preds, list):
        # Multi-stage outputs
        for stage_preds in preds:
            if isinstance(stage_preds, list):
                # Multi-scale outputs within a stage
                for scale_idx, pred in enumerate(stage_preds):
                    weight = 0.5 ** (num_scales - scale_idx - 1)
                    tgt_resized = resize_target_to(pred, targets)
                    loss = jnp.mean(jnp.abs(pred - tgt_resized))
                    total_loss += weight * loss
            else:
                tgt_resized = resize_target_to(stage_preds, targets)
                total_loss += jnp.mean(jnp.abs(stage_preds - tgt_resized))
    else:
        tgt_resized = resize_target_to(preds, targets)
        total_loss = jnp.mean(jnp.abs(preds - tgt_resized))

    return total_loss

def compute_loss_moe(preds, targets):
    tgt = resize_target_to(preds, targets)
    return jnp.mean(jnp.abs(preds - tgt))

# Wrap at module scope so it compiles only once
def forward_apply(params, batch_stats, x, rngs):
    if batch_stats is not None:
        (preds, router_probs), updates = model.apply(
            {"params": params, "batch_stats": batch_stats},
            x,
            train=True,
            rngs=rngs,
            mutable=["batch_stats"],
        )
        return (preds, router_probs), updates["batch_stats"]
    else:
        return model.apply({"params": params}, x, train=True, rngs=rngs), None

@functools.partial(jax.jit, donate_argnums=(0,))
def train_step(state: TrainState, batch_input, batch_target, num_scales, rng):
    """Single training step (OOM-safe)."""

    def loss_fn(params, batch_stats, x, y, rng):
        rngs = {"dropout": rng}

        # Recompute activations in backward mode instead of storing to save memory
        (preds, router_probs), new_batch_stats = jax.checkpoint(forward_apply)(
            params, batch_stats, x, rngs
        )

        task_loss = compute_loss_moe(preds, y)
        entropy_loss = router_entropy_loss(router_probs)
        balance_loss = load_balance_loss(router_probs)
        psnr = compute_psnr(preds, y)
        lambda_entropy = 1e-3
        lambda_balance = 1e-2

        total_loss = (
            task_loss
            - lambda_entropy * entropy_loss
            + lambda_balance * balance_loss
        )
        aux = {
            "psnr": psnr,
            "entropy": entropy_loss,
            "balance": balance_loss,
            "router_usage": jnp.mean(router_probs, axis=0),
            "batch_stats": new_batch_stats,
        }

        return total_loss, aux

    # Don’t redeclare grad_fn every step → avoids huge graph buildup
    (loss, aux), grads = jax.value_and_grad(
        loss_fn, has_aux=True
    )(state.params, state.batch_stats, batch_input, batch_target, rng)

    # Apply gradients safely, TrainState remains intact
    new_state = state.apply_gradients(grads=grads)

    # Update batch stats explicitly if present
    if aux["batch_stats"] is not None:
        new_state = new_state.replace(batch_stats=aux["batch_stats"])

    metrics = {
        "loss": loss,
        "psnr": aux["psnr"],
        "entropy": aux["entropy"],
        "balance": aux["balance"],
        "router_usage": aux["router_usage"],
    }
    
    return new_state, metrics


In [ ]:
def recover_tree(keys, values):
    """Recovers a tree as a nested dict from flat names and values.

    This function is useful to analyze checkpoints that are saved by our programs
    without need to access the exact source code of the experiment. In particular,
    it can be used to extract an reuse various subtrees of the scheckpoint, e.g.
    subtree of parameters.
    Args:
      keys: a list of keys, where '/' is used as separator between nodes.
      values: a list of leaf values.
    Returns:
      A nested tree-like dict.
    """
    tree = {}
    sub_trees = collections.defaultdict(list)
    for k, v in zip(keys, values):
        if "/" not in k:
            tree[k] = v
        else:
            k_left, k_right = k.split("/", 1)
            sub_trees[k_left].append((k_right, v))
    for k, kv_pairs in sub_trees.items():
        k_subtree, v_subtree = zip(*kv_pairs)
        tree[k] = recover_tree(k_subtree, v_subtree)
    return tree


def get_params(ckpt_path):
    """Get params checkpoint."""

    with tf.io.gfile.GFile(ckpt_path, "rb") as f:
        data = f.read()
    values = np.load(io.BytesIO(data))
    params = recover_tree(*zip(*values.items()))
    params = params["opt"]["target"]

    return params


def load_maxim_backbone(params_moe, params_maxim):
    moe = unfreeze(params_moe)
    flat_moe = flatten_dict(moe)
    flat_maxim = flatten_dict(params_maxim)

    for k, v in flat_maxim.items():
        if k in flat_moe and flat_moe[k].shape == v.shape:
            flat_moe[k] = v

    return freeze(unflatten_dict(flat_moe))

In [ ]:
maxim_mod = importlib.import_module("maxim.models.maxim")
router_mod = importlib.import_module("maxim.models.router")
moe_mod = importlib.import_module("maxim.models.moe")

maxim_configs = ml_collections.ConfigDict(_MODEL_CONFIGS)
maxim_configs.variant = _MODEL_VARIANT_DICT[TASK]
maxim_model = maxim_mod.Model(**maxim_configs)

router = router_mod.RouterModel()
experts = [
    moe_mod.ExpertHead(),  # denoise
    moe_mod.ExpertHead(),  # deblur
    moe_mod.ExpertHead(),  # derain
    moe_mod.ExpertHead(),  # dehaze
    moe_mod.ExpertHead(),  # enhance
]
moe_model = moe_mod.MaximMoE(maxim_model, router, experts)

x = jnp.zeros((1, 256, 256, 3))
rng = random.PRNGKey(SEED)
params_maxim = get_params(CKPT_PATH)
variables = moe_model.init(rng, x, train=True)
params_moe = variables["params"]
batch_stats = variables.get("batch_stats")

params_moe = load_maxim_backbone(params_moe, params_maxim)
flat_maxim = flatten_dict(params_maxim)
final_key = 'stage_1_output_conv_0'
final_kernel = flat_maxim[(final_key, "kernel")]
final_bias = flat_maxim[(final_key, "bias")]

params_moe = unfreeze(params_moe)
for e in range(NUM_EXPERTS):
    params_moe[f"experts_{e}"]["output_conv"]["kernel"] = final_kernel
    params_moe[f"experts_{e}"]["output_conv"]["bias"] = final_bias

params_moe = freeze(params_moe)

y_maxim = maxim_model.apply({"params": params_maxim}, x, train=False)
y_maxim = y_maxim[-1][-1]
y_moe, gates = moe_model.apply({"params": params_moe, 'batch_stats': batch_stats}, x, train=False)

diff = jnp.mean(jnp.abs(y_maxim - y_moe))
print(diff)
print(jnp.mean(gates, axis=0))

maxim features shape: (1, 256, 256, 32)
expert outputs shape: (1, 5, 256, 256, 3)
maxim features shape: (1, 256, 256, 32)
expert outputs shape: (1, 5, 256, 256, 3)
0.048531078
[[[[0.2]]]


 [[[0.2]]]


 [[[0.2]]]


 [[[0.2]]]


 [[[0.2]]]]
